In [1]:
#!pip install langchain langchain-core langchain_community langchain_openai
#Using langchain for templates
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate

In [2]:
#If numpy 2.2.6 (which might show as warning/error on execution of prev cell)
#!pip uninstall numpy -y
#!pip install "numpy<2"

In [3]:
#In case transformers is 4.x as done earlier
#!pip install tokenizers==0.15.2
#!pip uninstall langchain-huggingface -y

In [2]:
#Using different models
from transformers import pipeline

In [ ]:
#downgrade protobuf to avoid warnings,if needed
#!pip install --upgrade protobuf==4.25.3
#restart session,if above step done
#!pip show protobuf

In [3]:
#Using smaller model
generator = pipeline("text2text-generation", model="google/flan-t5-large")

def get_completion(prompt):
    response = generator(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())

print(get_completion("What is a DEFI in context of crypto world"))
#Using different variant i.e larger model, to run remove """ """

Device set to use cpu


decentralized exchange for fiat money


In [4]:
#Back to Chain Approach & using templates
import torch
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
# Load the model and tokenizer locally
model_name = "google/flan-t5-large"  # You can also use "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

client = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, 
    torch_dtype=torch.float32,  # Uses lower precision for efficiency
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

Device set to use cpu


In [5]:
def get_completion(prompt):
    response = client(prompt)
    return (response[0]["generated_text"].strip())

print(get_completion("What is a DEFI in context of crypto world"))

decentralized exchange for fiat money


In [6]:
prompt = "What is crypto currency"
response = client(prompt)
print(response)

[{'generated_text': 'crypto currency'}]


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline

template2 = "Please write a {length} review,of the book {book_title}. "
input_variables2 = [ "length", "book_title" ]
prompt = PromptTemplate(
    input_variables=input_variables2,
    template=template2
)
#To check
#formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
#print(formatted_prompt)
llm = HuggingFacePipeline(pipeline=client)

#Chaining
chain_new = prompt | llm
response1 = chain_new.invoke({
    "length": "short",
    "book_title": "House of Dragon"
})
print('response1-chain: ',response1)

#Passing prompt directly into Pipeline
prompt = template2.format(
    length="short",
    book_title="House of Dragon"
)
response2 = client(prompt)
print('response2-direct: ', response2[0]["generated_text"])

#Passing prompt into Function
prompt = template2.format(
    length="short",
    book_title="Sherlock Holmes"
)
response3 = get_completion(prompt)
print('response3-func: ',response3)


response1-chain:  's a spooky tale of dragons , witches , and witchcraft . . .
response2-direct:  's a spooky tale of dragons , witches , and witchcraft . . .
response3-func:  's a spooky tale of dragons , witches , and witchcraft . . .


In [8]:
#Using function as defined above
formatted_prompt = prompt.format(length = "short", book_title = " House Of Dragon")
def get_completion(prompt):
    response = client(prompt,max_new_tokens=100, do_sample=False)
    return (response[0]["generated_text"].strip())
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response)

AI Response:
<class 'str'>
's a spooky tale of dragons , witches , and witchcraft . . .


#### Using Other Models
- Mistral

In [2]:
#from transformers import pipeline
#Note**Access to model mistralai/Mistral-7B-Instruct-v0.1 is restricted. You must have access to it and be
#authenticated to access it. If yes, then we can use the code below.
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run
"""
generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")


def get_completion(prompt):
    # For instruction-tuned models, prepend with an instruction-style format
    instruction = f"<s>[INST] {prompt} [/INST]"
    response = generator(instruction, max_new_tokens=100, do_sample=False)
    return response[0]["generated_text"].split("[/INST]")[-1].strip()

print(get_completion("What is defi in context of crypto world?"))
"""

'\ngenerator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")\n\n\ndef get_completion(prompt):\n    # For instruction-tuned models, prepend with an instruction-style format\n    instruction = f"<s>[INST] {prompt} [/INST]"\n    response = generator(instruction, max_new_tokens=100, do_sample=False)\n    return response[0]["generated_text"].split("[/INST]")[-1].strip()\n\nprint(get_completion("What is defi in context of crypto world?"))\n'

- Falcon

In [ ]:
##using Another heavier model
#Note ** this will download large tensors,configs etc.. for this model, thus to run remove """ """ & then run
"""
# Load a text-generation pipeline with an instruction-tuned model
generator = pipeline("text-generation", model="tiiuae/falcon-7b-instruct")

def get_completion(prompt):
    # For instruction-tuned models, prepend with an instruction-style format
    instruction = f"<s>[INST] {prompt} [/INST]"
    response = generator(instruction, max_new_tokens=100, do_sample=False)
    return response[0]["generated_text"].split("[/INST]")[-1].strip()

print(get_completion("What is defi in context of crypto world?"))
"""

##### Offline/Online mode - custom location for models
###### Download Model to a Custom Location (D: Drive)

In [4]:
#Set the TRANSFORMERS_CACHE environment variable before importing transformers
#Ignoring these for now,as we are doing this in cells below
#import os
#os.environ["TRANSFORMERS_CACHE"] = "D:\\huggingface"   # Windows
# os.environ["TRANSFORMERS_CACHE"] = "/d/huggingface" # Git Bash / WSL
#import torch
#from transformers import pipeline

In [5]:
#Alternatively, pass cache_dir directly in the pipeline call (more explicit and portable but sometimes depends on models):
#Both approaches work — cache_dir in the call overrides the env variable if both are set.
# generator = pipeline(
#     "text-generation",
#     model="tiiuae/falcon-7b-instruct",
#     cache_dir="D:\\huggingface_models"
# )

In [6]:
#So we can..
'''
Falcon-7b-instruct is ~14GB on disk and needs ~16GB RAM/VRAM. 
If you're on CPU only, generation will be very slow — consider tiiuae/falcon-rw-1b for testing.
See example below
'''
# generator = pipeline(
#     "text-generation",
#     model="tiiuae/falcon-7b-instruct",
#     torch_dtype=torch.bfloat16,         # saves memory; use float32 if issues arise
#     trust_remote_code=True,             # required for Falcon
#     device_map="auto",                  # auto GPU/CPU placement
#     cache_dir="D:\\huggingface_models"   # explicit, overrides env var too
# )

"\nFalcon-7b-instruct is ~14GB on disk and needs ~16GB RAM/VRAM. \nIf you're on CPU only, generation will be very slow — consider tiiuae/falcon-rw-1b for testing.\nSee example below\n"

In [7]:
# If downloaded then Test it
# def get_completion(prompt, max_new_tokens=200):
#     instruction = f"User: {prompt}\nFalcon:"   # Falcon instruct format
#     response = generator(
#         instruction,
#         max_new_tokens=max_new_tokens,
#         do_sample=False,
#         eos_token_id=generator.tokenizer.eos_token_id,
#         pad_token_id=generator.tokenizer.eos_token_id  # avoids padding warning
#     )
#     generated = response[0]["generated_text"]
#     # Strip the prompt prefix, return only the model's reply
#     return generated[len(instruction):].strip()

# print(get_completion("What is DeFi in the context of the crypto world?"))

In [8]:
#consider tiiuae/falcon-rw-1b
#falcon-rw-1b: a base model, not instruct-tuned, so we need to adjust the prompt format — 
#drop the User:/Falcon: wrapper and just pass the prompt directly:
'''
Download is only ~2.5GB, much faster
Since it's a base model, answers will be more like text continuation than clean Q&A responses
'''
import os
cache = r"D:\\huggingface"

os.environ["HF_HOME"] = cache
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(cache, "hub")
os.environ["HF_MODULES_CACHE"] = os.path.join(cache, "modules")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(cache, "transformers")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [9]:
#Check paths
import os

print("HF_HOME:", os.environ.get("HF_HOME"))
print("HF_MODULES_CACHE:", os.environ.get("HF_MODULES_CACHE"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

HF_HOME: D:\\huggingface
HF_MODULES_CACHE: D:\\huggingface\modules
TRANSFORMERS_CACHE: D:\\huggingface\transformers


In [10]:
#To check if notebook or virtual environment already has cache locations configured
import os
import transformers
import huggingface_hub

print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

for key in [
    "HF_HOME",
    "HUGGINGFACE_HUB_CACHE",
    "HF_MODULES_CACHE",
    "TRANSFORMERS_CACHE",
    "XDG_CACHE_HOME",
]:
    print(f"{key} =", os.environ.get(key))

e:\Lesson_2_demos\venv\lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


transformers: 4.55.2
huggingface_hub: 0.34.4
HF_HOME = D:\\huggingface
HUGGINGFACE_HUB_CACHE = D:\\huggingface\hub
HF_MODULES_CACHE = D:\\huggingface\modules
TRANSFORMERS_CACHE = D:\\huggingface\transformers
XDG_CACHE_HOME = None


In [ ]:
#If model downloaded (& Note: transformers is lower than 4.55 such as 4.37, then we can test this)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained(
    "tiiuae/falcon-rw-1b",
    #cache_dir=cache,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-rw-1b",
    #cache_dir=cache,
    trust_remote_code=True,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

def get_completion(prompt, max_new_tokens=200):
    response = generator(
        prompt,                          # no instruction wrapper needed
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )
    generated = response[0]["generated_text"]
    return generated[len(prompt):].strip()   # strip the input prompt from output

In [11]:
#Check
from pathlib import Path
root = Path(r"D:\\huggingface")
for p in root.rglob("model.safetensors"):
    print(p)

D:\huggingface\transformers\models--tiiuae--falcon-rw-1b\.no_exist\e4b9872bb803165eb22f0a867d4e6a64d34fce19\model.safetensors


In [ ]:
#If all good and downloaded
print(get_completion("What is DeFi in the context of the crypto world?"))
#If errors, try next cell too

In [ ]:
#Testing compatibility of falcon with transformers
#This may take lot of time and still not work as falcon with transformers-4.55 may have compatibiltiy issue
#so try it and if it takes time, interrupt kernel

import torch

prompt = "What is DeFi in the context of the crypto world?"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=False,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
from huggingface_hub import scan_cache_dir
info = scan_cache_dir(r"D:\huggingface\transformers")
print(info)
#might show: corruption error

HFCacheInfo(size_on_disk=0, repos=frozenset(), warnings=[CorruptedCacheException("Reference(s) refer to missing commit hashes: {'79322c51ba0241c81368bdeb9c6795398aeeb22e': {'refs\\\\pr\\\\7'}} (D:\\huggingface\\transformers\\models--tiiuae--falcon-rw-1b).")])


### Switching to usage of bigger LLMs deployed on endpoints

In [9]:
#repeating previous steps
template2 = "Please write a {length} review,of the book {book_title}. "
input_variables2 = [ "length", "book_title" ]

prompt = template2.format(
    length="short",
    book_title="House of Dragon"
)

#Using OpenAI and gpt model
#Note** If using gpt model and AzureOpenAI or AzureChatOpenAI (refer: 'Working_with_AzureOpenAI' folder)
import openai
import os
from openai import AzureOpenAI
# Initialize client once
from dotenv import load_dotenv
#load_dotenv("/content/.env")
load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
)
deployment_name = os.getenv("AZURE_DEPLOYMENT_NAME")
#or
'''
#Using Langchain Equivalent
from langchain_openai import AzureChatOpenAI

#from dotenv import load_dotenv
#load_dotenv("/content/.env")
#load_dotenv()

client = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("API_KEY"),
    api_version="2024-12-01-preview",
    deployment_name="gpt-4.1",
    temperature=0,
)

client.invoke("Explain transformers in simple terms in 25 words").content
'''

'\n#Using Langchain Equivalent\nfrom langchain_openai import AzureChatOpenAI\n\n#from dotenv import load_dotenv\n#load_dotenv("/content/.env")\n#load_dotenv()\n\nclient = AzureChatOpenAI(\n    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),\n    api_key=os.getenv("API_KEY"),\n    api_version="2024-12-01-preview",\n    deployment_name="gpt-4.1",\n    temperature=0,\n)\n\nclient.invoke("Explain transformers in simple terms in 25 words").content\n'

In [10]:
#Creating function
def get_completion(prompt, deployment_name=deployment_name):
    """Get a chat completion from Azure OpenAI.
    Args:
        prompt (str): User input prompt.
        deployment_name (str): The deployment name you gave your model in Azure portal.
    Returns:
        dict: Full response object, or error dict.
    """
    try:
        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=deployment_name,    # <-- This is the "deployment name" not the raw model name
            messages=messages,
            temperature=0.1,
            top_p=0.8,
            max_tokens=512
        )
        #return response.model_dump()  # Return the full response as dict
        # Extract just the assistant's reply
        return response.choices[0].message.content
    except Exception as e:
        return {"error": str(e)}

In [11]:
response = get_completion(prompt)
response

'Certainly! However, it’s worth noting that **"House of the Dragon"** is primarily known as a television series, a prequel to *Game of Thrones*, rather than a book. The show is based on George R.R. Martin’s book **"Fire & Blood"**, which chronicles the history of House Targaryen.\n\nIf you meant **"Fire & Blood"**, here’s a short review:\n\n---\n\n**Review of "Fire & Blood" by George R.R. Martin**\n\n*"Fire & Blood"* is a sweeping chronicle of the Targaryen dynasty, set centuries before the events of *A Song of Ice and Fire*. Written in the style of a historical account, the book delves into the rise and fall of Targaryen kings, their dragons, and the bloody intrigues that shaped Westeros. Martin’s storytelling is rich and detailed, though the narrative can feel dense and less character-driven than his main series. Fans of Westerosi lore will relish the political machinations, epic battles, and the origins of the infamous Dance of the Dragons. While it reads more like a history textboo

In [ ]:
#If using get_completion() based on gpt model as defined above, then we can
"""
response = get_completion(formatted_prompt)
print("AI Response:")
print(type(response))
print(response.keys())

response['choices'][0]['message']['content']"""

#### Formatting
- ##### Jinja template

In [21]:
#Jinja Template example
jinja2_template = "Give me an {{ adjective }} fact about {{ topic }}"

In [22]:
prompt = PromptTemplate.from_template(jinja2_template, template_format = "jinja2" )

In [23]:
user_question = prompt.format(adjective="interesting", topic="space exploration")
print(user_question)

Give me an interesting fact about space exploration


In [24]:
response = get_completion(user_question)
print("AI Response:")
print(response)

AI Response:
Sure! Did you know that **the footprints left by astronauts on the Moon could last for millions of years**? That’s because the Moon has no atmosphere, so there’s no wind or water to erode or wash away the marks. The only things that might disturb them are meteorite impacts!


- ##### F-string

In [25]:
#Using f-string (example)
fstring_template = "Here is a brief summary for the book titled '{book_title}':"
book_title = "The Great Gatsby"
prompt = fstring_template.format(book_title=book_title)

In [ ]:
#Testing a dummy function
def get_book_summary(prompt):
    return "It's a novel about love, wealth, and aspiration, set in the Roaring '20s."

In [ ]:
summary = get_book_summary(prompt)
print(summary)

It's a novel about love, wealth, and aspiration, set in the Roaring '20s.


In [26]:
#Using function that invokes the LLM
response = get_completion(prompt)
print("AI Response:")
print(response)

AI Response:
Certainly! Please provide your summary of "The Great Gatsby," and let me know how I can assist you with it.


In [27]:
# Example where string prompt template would not work
# Define the prompt template

jinja2_template = """

Dear {{ name }},
{% if age < 18 %}
You are invited to our kids' event with activities such as face painting, bouncy castles, and clown shows.
{% elif age < 65 %}
You are invited to our adult event with activities like live music, wine tasting, and art workshops.
{% else %}
You are invited to our senior event with activities including book clubs, chess tournaments, and tea dances.
{% endif %}
Sincerely,
Event Organizer

Write the mail in 200 words
"""

In [30]:
prompt = PromptTemplate.from_template(jinja2_template, template_format="jinja2")

# Format the prompt with specific values for 'action', 'group', and 'time_period'
argument_prompt = prompt.format(name="John Doe", age=55)

In [31]:
response = get_completion(argument_prompt)
print("AI Response:")
print(response)

AI Response:
Subject: Invitation to Our Exclusive Adult Event – An Evening of Art, Wine, and Music

Dear John Doe,

We are delighted to invite you to an exclusive adult evening designed to inspire, entertain, and connect. Join us for a memorable event featuring live music performances, curated wine tasting sessions, and engaging art workshops. This is the perfect opportunity to unwind, explore your creativity, and enjoy the company of fellow enthusiasts in a vibrant and welcoming atmosphere.

Our talented musicians will set the mood with a selection of live performances throughout the evening. Wine connoisseurs will guide you through a tasting of exquisite local and international wines, offering insights into each vintage’s unique character. For those who love to create, our art workshops will provide hands-on experiences led by skilled artists, suitable for all levels of expertise.

The event will take place on [Date] at [Venue], starting at [Time]. Dress code is smart casual. All act

#### Using Langchain templates & chat mode

In [32]:
simple_prompt = "The {subject} is strong in this one."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [33]:
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [34]:
simple_message_template = HumanMessagePromptTemplate.from_template(simple_prompt)
human_message_template = HumanMessagePromptTemplate.from_template(human_prompt)

chat_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="conversation"),
    simple_message_template,
    human_message_template
])

In [35]:
human_message = HumanMessage(content="What's the best way to learn a new language?")
ai_message = AIMessage(content="""\
1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.
2. Practice regularly: Consistency is key when learning a new language.
3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.\
""")

In [36]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Force",
    word_count="10"
).to_messages()

In [37]:
print(conversation)

[HumanMessage(content="What's the best way to learn a new language?", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Immerse yourself in the language: Try to use the language in your daily life as much as possible.\n2. Practice regularly: Consistency is key when learning a new language.\n3. Use language learning apps: There are many apps that can help you learn a new language in a fun and engaging way.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='The Force is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [38]:
simple_prompt = "The {subject} is fascinating to study."
human_prompt = "Summarize our conversation so far in {word_count} words."

In [39]:
human_message = HumanMessage(content="What's happens inside a black hole")
ai_message = AIMessage(content="""\
1. Inside black hole gravity is zero way.\
""")

In [40]:
conversation = chat_prompt.format_prompt(
    conversation=[human_message, ai_message],
    subject="Black Hole",
    word_count="10"
).to_messages()

In [41]:
prompt_string = conversation
print("Formatted Prompt:")
print(prompt_string)


Formatted Prompt:
[HumanMessage(content="What's happens inside a black hole", additional_kwargs={}, response_metadata={}), AIMessage(content='1. Inside black hole gravity is zero way.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='The Black Hole is strong in this one.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarize our conversation so far in 10 words.', additional_kwargs={}, response_metadata={})]


In [ ]:
print(type(prompt_string))

<class 'list'>


In [ ]:
#To be fixed in format
#response = get_completion(prompt_string)
#print(response)

In [ ]:
#Using Langchain templates & styles

In [42]:
#Using style
#Modify parameters like **customer_style** and **customer_email**
#to influence the tone and formality of the generated responses.

template_string = """Translate the text that is delimited by triple backticks into a style
that is {style}. text: ```{text}```"""

# Style and email input
customer_style = "American English in a casual tone"
customer_email = """
I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!
"""

In [43]:
prompt = template_string.format(style=customer_style, text=customer_email)

In [44]:
instruction_prompt = f"<s>[INST] {prompt} [/INST]"

In [45]:
#Using generator based on google/flan-t5-base
response = generator(instruction_prompt, max_new_tokens=150, do_sample=False)

In [46]:
response

[{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!  [/INST]"}]

In [47]:
generated_text = response[0]['generated_text'].split("[/INST]")[-1].strip()

In [48]:
print(response[0])

{'generated_text': " I'm super excited about the new gaming console I bought! It arrived in just 2 days and I've been playing non-stop. Totally worth the price!  [/INST]"}


In [50]:
response = get_completion(prompt)
print(response)

I'm really pumped about the new gaming console I got! It showed up in just two days, and I’ve been playing it nonstop. Definitely worth the money!


In [51]:
response = get_completion(instruction_prompt)
print(response)

I'm so pumped about the new gaming console I got! It showed up in just two days, and I’ve been playing nonstop. Totally worth every penny!


In [57]:
#to be checked...
template_string = """Translate the text that is delimited by triple backticks into a style that is {style}.
text: ```{text}```"""

'''
prompt_template = ChatPromptTemplate.from_template(template_string)
message = prompt_template.format_messages(
    style="Scottish English in a professional tone",
    text="I'm super excited about the new gaming console & new game!"
)'''

style="Scottish english in a professional tone"
text="I'm super excited about the new gaming console & new game!"


prompt = template_string.format(style=style, text=text)

#Using generator based on google/flan-t5-base
#response = generator(prompt,max_new_tokens=150, do_sample=False)
response = get_completion(prompt)
print(response)

I’m absolutely delighted about the new gaming console and the new game!
